# Sort Recordings by Environment (TMaze vs Linear)

This notebook classifies recording sessions into **TMaze** or **Linear** environments by analyzing a frame from the `Linear*` video file in each session. It then moves (or copies) all four associated files into the appropriate subfolder.

## How it works
Each session has four files sharing the same timestamp suffix:
- `Linear<timestamp>.avi` — main arena camera (used for classification)
- `Home<timestamp>.avi`
- `Miniscope<timestamp>.avi`
- `timestamps<timestamp>.csv`

The classifier extracts one frame from `Linear*.avi` and detects bright white track structures against the dark background. It then measures **aspect ratio and shape** of the detected region:
- **TMaze**: cross/plus-shaped → roughly equal width & height
- **Linear**: two parallel horizontal arms → much wider than tall


In [ ]:
# ── Dependencies ────────────────────────────────────────────────────────────
import subprocess, sys
for pkg in ['opencv-python-headless', 'numpy', 'matplotlib', 'tqdm']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('All dependencies ready.')

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import shutil
from pathlib import Path
from collections import defaultdict
from tqdm.notebook import tqdm

print('Imports OK')

## ⚙️  Configuration — edit this cell

In [ ]:
# ── USER SETTINGS ────────────────────────────────────────────────────────────

# Root folder containing all Mouse*** subdirectories
ROOT_DIR = Path('/media/toor/Seagate3/YOUNG_MICE/Mouse945')

# Set to True to MOVE files, False to COPY (safer for first run)
MOVE_FILES = True
# Frame index to extract for classification (0 = first frame; increase if
# the first frames are dark/blank, e.g. 100 or 500)
FRAME_INDEX = 500

# Brightness threshold for isolating the white track (0-255).
# Raise if the arena floor is bright; lower if the track is dim.
BRIGHTNESS_THRESHOLD = 110

# Aspect-ratio cut-off: bounding-box width/height of the track blob.
# Values > this → Linear; values <= this → TMaze.
# The T-maze is roughly square (ratio ≈ 1), the linear track is wide (ratio > 2).
ASPECT_RATIO_CUTOFF_UPPER = 1.85
ASPECT_RATIO_CUTOFF_LOWER = 1.53

# ── END OF USER SETTINGS ─────────────────────────────────────────────────────

## Step 1 — Discover sessions

In [ ]:
def discover_sessions(root: Path):
    """
    Walk every Mouse*** subdirectory and group files by their shared timestamp
    suffix (everything after the prefix word, e.g. '2025-12-18T10_29_21').

    Returns a dict:
        { (mouse_dir, timestamp): {'Linear': Path, 'Home': Path,
                                    'Miniscope': Path, 'timestamps': Path} }
    Only sessions where the Linear .avi exists are kept.
    """
    # Regex: capture the prefix word and the timestamp
    pattern = re.compile(
        r'^(Linear|Home|Miniscope|timestamps)(\d{4}-\d{2}-\d{2}T\d{2}[_:]\d{2}[_:]\d{2})'
        r'\.(avi|csv)$', re.IGNORECASE
    )

    sessions = defaultdict(dict)  # (mouse_dir, ts) → {prefix: Path}

    mouse_dirs = sorted(p for p in root.iterdir()
                        if p.is_dir() and re.match(r'Mouse', p.name, re.I))

    if not mouse_dirs:
        # Fallback: no Mouse* sub-dirs → treat root itself as the only folder
        mouse_dirs = [root]

    for mdir in mouse_dirs:
        for f in mdir.iterdir():
            m = pattern.match(f.name)
            if m:
                prefix, ts, _ = m.groups()
                key = (mdir, ts)
                sessions[key][prefix.capitalize()] = f

    # Keep only sessions that have the Linear video
    valid = {k: v for k, v in sessions.items() if 'Linear' in v}
    print(f'Found {len(valid)} sessions across {len(mouse_dirs)} mouse folder(s).')
    return valid


sessions = discover_sessions(ROOT_DIR)

# Preview
for (mdir, ts), files in list(sessions.items()):
    print(f'  {mdir.name} / {ts}  →  {list(files.keys())}')

## Step 2 — Classifier

In [ ]:
def extract_frame(video_path: Path, frame_index: int = 100) -> np.ndarray | None:
    """Return a single BGR frame from a video, or None on failure."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idx = min(frame_index, max(0, total - 1))
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    cap.release()
    return frame if ret else None


def classify_frame(frame: np.ndarray,
                   brightness_threshold: int = 180,
                   aspect_ratio_cutoff_upper: float = 1.8,
                   aspect_ratio_cutoff_lower: float = 2.5,
                   debug: bool = False) -> str:
    """
    Classify a frame as 'Linear' or 'TMaze'.

    Strategy
    --------
    1. Convert to grayscale.
    2. Apply fisheye mask (circular ROI) to ignore the black border.
    3. Threshold to isolate the bright white track.
    4. Morphologically close small gaps.
    5. Find the largest contour and compute its bounding-box aspect ratio.
       - Linear track: very wide relative to height  → ratio >> 1
       - TMaze:        roughly equal width and height → ratio ≈ 1
    """
    h, w = frame.shape[:2]
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # 1. Circular fisheye mask
    mask = np.zeros((h, w), dtype=np.uint8)
    cx, cy = w // 2, h // 2
    radius = int(min(cx, cy) * 0.99)   # slightly inset from the vignette edge
    cv2.circle(mask, (cx, cy), radius, 255, -1)

    gray_masked = cv2.bitwise_and(gray, gray, mask=mask)

    # 2. Threshold
    _, binary = cv2.threshold(gray_masked, brightness_threshold, 255,
                              cv2.THRESH_BINARY)

    # 3. Morphological close to fill track interior
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

    # 4. Find contours, pick the largest
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return 'UNKNOWN'

    largest = max(contours, key=cv2.contourArea)
    x, y, bw, bh = cv2.boundingRect(largest)
    aspect_ratio = bw / bh if bh > 0 else 0

    # Optional: convexity / solidity check for extra robustness
    # hull_area = cv2.contourArea(cv2.convexHull(largest))
    # solidity = cv2.contourArea(largest) / hull_area if hull_area > 0 else 0

    label = 'Linear' if aspect_ratio > aspect_ratio_cutoff_lower and aspect_ratio < aspect_ratio_cutoff_upper else 'TMaze'

    if debug:
        print(f'    bounding box: {bw}×{bh}  →  aspect ratio = {aspect_ratio:.2f}  →  {label}')

    return label


def classify_session(linear_video: Path,
                     frame_index: int = FRAME_INDEX,
                     brightness_threshold: int = BRIGHTNESS_THRESHOLD,
                     aspect_ratio_cutoff_upper: float = ASPECT_RATIO_CUTOFF_UPPER,
                     aspect_ratio_cutoff_lower: float = ASPECT_RATIO_CUTOFF_LOWER,
                     debug: bool = False) -> str:
    """Top-level: extract a frame from the Linear video and classify it."""
    frame = extract_frame(linear_video, frame_index)
    if frame is None:
        return 'UNKNOWN'
    return classify_frame(frame, brightness_threshold, aspect_ratio_cutoff_upper, aspect_ratio_cutoff_lower, debug)


print('Classifier functions defined.')

## Step 3 — Visual spot-check (optional but recommended)

Inspect a handful of frames to verify the threshold settings before sorting.

In [ ]:
# Show up to 6 sample frames with thir predicted labels
sample_keys = list(sessions.keys())[80:95]

fig, axes = plt.subplots(2, len(sample_keys), figsize=(4 * len(sample_keys), 6))
if len(sample_keys) == 1:
    axes = np.array(axes).reshape(2, 1)

for col, key in enumerate(sample_keys):
    mdir, ts = key
    linear_path = sessions[key]['Linear']
    frame = extract_frame(linear_path, FRAME_INDEX)
    if frame is None:
        continue

    label = classify_frame(frame, BRIGHTNESS_THRESHOLD, ASPECT_RATIO_CUTOFF_UPPER, ASPECT_RATIO_CUTOFF_LOWER, debug=True)

    # Top row: original frame (RGB)
    axes[0, col].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axes[0, col].set_title(f'{mdir.name}\n{ts[:10]}', fontsize=8)
    axes[0, col].axis('off')

    # Bottom row: binary mask
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, BRIGHTNESS_THRESHOLD, 255, cv2.THRESH_BINARY)
    axes[1, col].imshow(binary, cmap='gray')
    color = 'green' if label != 'UNKNOWN' else 'red'
    axes[1, col].set_title(f'→  {label}', fontsize=9,
                           fontweight='bold', color=color)
    axes[1, col].axis('off')

plt.suptitle('Top: raw frame  |  Bottom: brightness mask  |  Predicted label', y=1.02)
plt.tight_layout()
plt.show()
print('\nIf any labels look wrong, adjust BRIGHTNESS_THRESHOLD or ASPECT_RATIO_CUTOFF in the Configuration cell and re-run.')

## Step 4 — Classify all sessions

In [ ]:
results = {}   # (mdir, ts) → 'Linear' | 'TMaze' | 'UNKNOWN'

for key, files in tqdm(sessions.items(), desc='Classifying sessions'):
    label = classify_session(files['Linear'],
                             frame_index=FRAME_INDEX,
                             brightness_threshold=BRIGHTNESS_THRESHOLD,
                             aspect_ratio_cutoff_upper=ASPECT_RATIO_CUTOFF_UPPER,aspect_ratio_cutoff_lower=ASPECT_RATIO_CUTOFF_LOWER)
    results[key] = label

# Summary
from collections import Counter
counts = Counter(results.values())
print('\nClassification summary:')
for label, n in sorted(counts.items()):
    print(f'  {label:10s}: {n} sessions')

if counts.get('UNKNOWN', 0):
    print('\n⚠️  Some sessions could not be classified. '
          'Check FRAME_INDEX or BRIGHTNESS_THRESHOLD.')

## Step 5 — Review before sorting (dry run)

Print every planned file operation without touching anything.

In [ ]:
PREFIX_KEYS = ['Linear', 'Home', 'Miniscope', 'Timestamps']

def plan_operations(sessions, results):
    """
    For each session, yield (src_path, dst_path) tuples.
    Destination is mdir / {TMaze|Linear} / filename.
    """
    ops = []
    for key, label in results.items():
        if label == 'UNKNOWN':
            continue
        mdir, ts = key
        dest_dir = mdir / label          # e.g.  .../Mouse01/TMaze/
        for pkey, src in sessions[key].items():
            dst = dest_dir / src.name
            ops.append((src, dst, dest_dir))
    return ops


ops = plan_operations(sessions, results)

print(f'Total file operations planned: {len(ops)}')
print(f'Action: {"MOVE" if MOVE_FILES else "COPY"}')
print()
for src, dst, _ in ops[:20]:   # show first 20
    print(f'  {src.parent.name}/{src.name}')
    print(f'    → {dst.parent.parent.name}/{dst.parent.name}/{dst.name}')
if len(ops) > 20:
    print(f'  … and {len(ops) - 20} more.')

## Step 6 — Execute

> **Set `MOVE_FILES = True` in the Configuration cell if you want to move instead of copy.**
> Run the spot-check (Step 3) and dry-run (Step 5) first to confirm labels.

In [ ]:
skipped, done, errors = [], [], []

for src, dst, dest_dir in tqdm(ops, desc=('Moving' if MOVE_FILES else 'Copying') + ' files'):
    if not src.exists():
        skipped.append(src)
        continue
    try:
        dest_dir.mkdir(parents=True, exist_ok=True)
        if MOVE_FILES:
            shutil.move(str(src), str(dst))
        else:
            shutil.copy2(str(src), str(dst))
        done.append(dst)
    except Exception as exc:
        errors.append((src, exc))

print(f'\n✅  Done:    {len(done)} files')
print(f'⚠️  Skipped: {len(skipped)} (source not found)')
print(f'❌  Errors:  {len(errors)}')
for src, exc in errors:
    print(f'   {src.name}: {exc}')

## Step 7 — Final report

In [ ]:
print('=== FINAL REPORT ===')
print()
# Per-mouse breakdown
per_mouse = defaultdict(lambda: Counter())
for (mdir, ts), label in results.items():
    per_mouse[mdir.name][label] += 1

for mouse, cnts in sorted(per_mouse.items()):
    parts = ', '.join(f'{lbl}: {n}' for lbl, n in sorted(cnts.items()))
    print(f'  {mouse:20s}  {parts}')

print()
print('UNKNOWN sessions (need manual review):')
unknowns = [(k, v) for k, v in results.items() if v == 'UNKNOWN']
if unknowns:
    for (mdir, ts), _ in unknowns:
        print(f'  {mdir.name}  /  {ts}')
else:
    print('  None — all sessions classified successfully! 🎉')